# Modify: limpieza y representaciones sin metadatos de etiqueta

VulnMiner · DavidPilaguano · SEMMA con datos públicos verificables.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / 'data/dataset.csv').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image
df = pd.read_csv(ROOT / 'data/dataset.csv', keep_default_na=False)
df['labels'] = df.labels.map(json.loads)
manifest = json.loads((ROOT / 'data/public_manifest.json').read_text(encoding='utf-8'))
results = json.loads((ROOT / 'output/evaluation/metrics.json').read_text(encoding='utf-8'))
selected = results['candidates'][results['selected']]


La importación conserva originales comprimidos y crea fragmentos preparados. Python usa AST para extraer el cuerpo del handler POST; Java elimina el wrapper GET, comentarios y encabezados. Se retiran IDs BenchmarkTest, rutas que contienen categorías y docstrings. Se conservan operaciones, condiciones, literales y controles que afectan al comportamiento.

In [2]:
from scripts.import_public_datasets import structural_fingerprint
assert df.groupby('project').split.nunique().max() == 1
from vulnminer.features import make_features
parts = {s: df[df.split == s] for s in ['train','validation','test']}
features = make_features()
X = {'train': features.fit_transform(parts['train'].code)}
X.update({s: features.transform(parts[s].code) for s in ['validation','test']})
print({s: matrix.shape for s,matrix in X.items()})

{'train': (3504, 26723), 'validation': (735, 26723), 'test': (735, 26723)}


In [3]:
from scipy.sparse import load_npz
for split in parts:
    matrix = load_npz(ROOT / 'output/evaluation' / f'features_{split}.npz')
    print(split, matrix.shape, 'no ceros:', matrix.nnz)
print('Solo code alimenta el modelo; source, CWE, labels e id no son características.')

train (3504, 26723) no ceros: 7387413
validation (735, 26723) no ceros: 1495663


test (735, 26723) no ceros: 1467097
Solo code alimenta el modelo; source, CWE, labels e id no son características.


TF-IDF combina caracteres 3..5 y tokens 1..2, con vocabulario e IDF ajustados únicamente sobre entrenamiento. Se usa class_weight=balanced. Las matrices, etiquetas y orden de filas se exportan. Las huellas de agrupación no se utilizan como predictores.